# setup + imports

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, f1_score

from sklearn.ensemble import HistGradientBoostingClassifier

In [7]:
import io
import zipfile
import numpy as np
import argparse
import os
from pathlib import Path
from baseline import _load_array, load_dataset, write_submission, predict, _best_threshold, _temporal_average

N_CLASSES = 29

In [8]:
folder = "nightwatch_data/"

data = {
    "train_x": np.load(folder + "train_embeddings.npy"),
    "train_y": np.load(folder + "train_labels.npy"),
    "valid_x": np.load(folder + "validation_embeddings.npy"),
    "valid_y": np.load(folder + "validation_labels.npy"),
    "test_x": np.load(folder + "test_embeddings.npy"),
}

X_train = data["train_x"].reshape(-1, data["train_x"].shape[-1]).astype(np.float64)
y_train = data["train_y"].reshape(-1, N_CLASSES).astype(bool)
X_val = data["valid_x"].reshape(-1, data["valid_x"].shape[-1]).astype(np.float64)
y_val = data["valid_y"].reshape(-1, N_CLASSES).astype(bool)
X_test = data["test_x"].reshape(-1, data["test_x"].shape[-1]).astype(np.float64)

y_val

array([[False, False, False, ..., False, False, False],
       [ True, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])

In [31]:
train_dom = np.load(folder + "train_domain_ids.npy")
val_dom = np.load(folder + "validation_domain_ids.npy")

print(np.unique(train_dom, return_counts=True))
print(np.unique(val_dom, return_counts=True))

(array([0, 1, 2], dtype=uint8), array([210, 220, 210]))
(array([0, 1, 2], dtype=uint8), array([47, 50, 63]))


# validations

In [22]:
def score(refs, preds):
    ap = []
    for i in range(N_CLASSES):
        ap.append(average_precision_score(refs[:, i], preds[:, i]))
    oldap = np.mean(ap[:16])
    newap = np.mean(ap[16:])
    f1 = f1_score(refs.reshape(-1), preds.reshape(-1)>0.5)
    total = 100 * (0.35 * oldap + 0.35 * newap + 0.30 * f1)
    print('old AP', round(oldap, 4), ' new AP', round(newap, 4), ' F1', round(f1, 4), ' total', round(total, 2))
    return total

# modele

In [23]:

models = []
valpreds = np.zeros((len(X_val), N_CLASSES))
for c in range(N_CLASSES):
    model = HistGradientBoostingClassifier()

    y_tr_c = y_train[:, c]
    y_val_c = y_val[:, c]
    model.fit(X_train, y_tr_c)

    valpreds[:, c] = model.predict_proba(X_val)[:, 1]

models.append(model)
s = score(y_val, valpreds)
print(s)

old AP 0.8451  new AP 0.5726  F1 0.7517  total 72.17
72.17127080060656


In [29]:
X_all = np.concatenate([X_train, X_val])
y_all = np.concatenate([y_train, y_val])


preds = np.zeros((len(X_test), N_CLASSES))
for c in range(N_CLASSES):
    model = HistGradientBoostingClassifier()

    y_c = y_all[:, c]
    model.fit(X_all, y_c)

    preds[:, c] = model.predict_proba(X_test)[:, 1]

preds = preds.reshape(360, 24, N_CLASSES)
print(preds.shape) 

(360, 24, 29)


### other tweaks:

In [32]:
## acum daca we leave one domain out:

dom_all = np.concatenate([train_dom, val_dom])         
X_all = np.concatenate([X_train, X_val])                
y_all = np.concatenate([y_train, y_val])               

dom_frame = []
for s in range(len(dom_all)):
    for t in range(24):
        dom_frame.append(dom_all[s])                   
dom_frame = np.array(dom_frame)                       

def train_predict(X_fit, y_fit, X_new):
    P = np.zeros((len(X_new), N_CLASSES))
    for c in range(N_CLASSES):
        model = HistGradientBoostingClassifier(random_state=0)
        model.fit(X_fit, y_fit[:, c])
        P[:, c] = model.predict_proba(X_new)[:, 1]
    return P

left out domain 0
old AP 0.847  new AP 0.5359  F1 0.7068  total 69.6
left out domain 1
old AP 0.8287  new AP 0.4656  F1 0.6763  total 65.59
left out domain 2
old AP 0.849  new AP 0.5205  F1 0.7247  total 69.67
mean 68.28846290345668


In [35]:
scores = []
for d in range(3):
    fit = dom_frame != d          # True for frames of the other two domains
    out = dom_frame == d          # True for frames of domain d
    P = train_predict(X_all[fit], y_all[fit], X_all[out])
    print('left out domain', d)
    scores.append(score(y_all[out], P))
print('mean', np.mean(scores))

left out domain 0
old AP 0.7416  new AP 0.3609  F1 0.5818  total 56.04
left out domain 1
old AP 0.7417  new AP 0.3105  F1 0.5733  total 54.03
left out domain 2
old AP 0.7516  new AP 0.4034  F1 0.5857  total 58.0
mean 56.02156154392895


In [34]:
def scenenorm(scene):
    out = np.zeros_like(scene)
    for i in range(len(scene)):
        m = scene[i].mean(axis=0)
        for t in range(24):
            out[i, t] = scene[i, t] - m
    return out

train_n = scenenorm(data["train_x"]).reshape(-1, 48)
val_n = scenenorm(data["valid_x"]).reshape(-1, 48)
X_all = np.concatenate([train_n, val_n])

In [38]:
def add_scene_diff(E):                    # E: (scenes, 24, 48) -> (scenes, 24, 96)
    norm = scenenorm(E)
    out = np.zeros((E.shape[0], E.shape[1], 96), dtype=E.dtype)
    out[:, :, :48] = E                    # first 48: raw
    out[:, :, 48:] = norm                 # last 48: minus the scene average
    return out

train_b = add_scene_diff(data["train_x"]).reshape(-1, 96)
val_b = add_scene_diff(data["valid_x"]).reshape(-1, 96)
X_all = np.concatenate([train_b, val_b])

scores = []
for d in range(3):
    fit = dom_frame != d          # True for frames of the other two domains
    out = dom_frame == d          # True for frames of domain d
    P = train_predict(X_all[fit], y_all[fit], X_all[out])
    print('left out domain', d)
    scores.append(score(y_all[out], P))
print('mean', np.mean(scores))

left out domain 0
old AP 0.8513  new AP 0.5102  F1 0.7111  total 68.99
left out domain 1
old AP 0.8554  new AP 0.4477  F1 0.6762  total 65.9
left out domain 2
old AP 0.8656  new AP 0.5283  F1 0.7348  total 70.83
mean 68.57147979691024


## LODO?? leave one domain out + threshld tuning

In [40]:
oof = np.zeros((len(X_all), N_CLASSES))      # for every frame: prediction from a model that never saw its domain
for d in range(3):
    fit = dom_frame != d
    out = dom_frame == d
    oof[out] = train_predict(X_all[fit], y_all[fit], X_all[out])

def lodo_score(P):
    scores = []
    for d in range(3):
        out = dom_frame == d
        scores.append(score(y_all[out], P[out]))
    print('mean', np.mean(scores))
    return np.mean(scores)

lodo_score(oof)                               

old AP 0.8513  new AP 0.5102  F1 0.7111  total 68.99
old AP 0.8554  new AP 0.4477  F1 0.6762  total 65.9
old AP 0.8656  new AP 0.5283  F1 0.7348  total 70.83
mean 68.57147979691024


68.57147979691024

In [41]:
def smooth(P):          # P: (scenes, 24, 29)
    out = np.zeros_like(P)
    T = P.shape[1]
    for t in range(T):
        prev = P[:, max(t - 1, 0)]
        nxt = P[:, min(t + 1, T - 1)]
        out[:, t] = 0.25 * prev + 0.5 * P[:, t] + 0.25 * nxt
    return out

oof_s = smooth(oof.reshape(-1, 24, N_CLASSES)).reshape(-1, N_CLASSES)
lodo_score(oof_s)

old AP 0.8451  new AP 0.5203  F1 0.6788  total 68.15
old AP 0.8482  new AP 0.4921  F1 0.6603  total 66.72
old AP 0.8703  new AP 0.5558  F1 0.7123  total 71.28
mean 68.71748487304478


68.71748487304478

In [43]:
for thr in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    f1 = f1_score(y_all.reshape(-1), oof_s.reshape(-1) > thr)
    print(thr, round(f1, 4))

thr = 0.25

def to_half(P, thr):
    out = np.zeros_like(P)
    low = P < thr
    out[low] = 0.5 * P[low] / thr                          # [0, thr) -> [0, 0.5)
    out[~low] = 0.5 + 0.5 * (P[~low] - thr) / (1 - thr)    # [thr, 1] -> [0.5, 1]
    return out

oof_h = to_half(oof_s, 0.25)
lodo_score(oof_h)

0.2 0.6953
0.25 0.7442
0.3 0.7425
0.35 0.7348
0.4 0.7279
0.45 0.7168
0.5 0.6831
old AP 0.8451  new AP 0.5203  F1 0.7765  total 71.08
old AP 0.8482  new AP 0.4921  F1 0.6986  total 67.87
old AP 0.8703  new AP 0.5558  F1 0.7682  total 72.96
mean 70.63727835891204


70.63727835891204

# submission

In [39]:
test_b = add_scene_diff(data["test_x"]).reshape(-1, 96)      # (8640, 96)

P_test = train_predict(X_all, y_all, test_b)                 # X_all = the 96-feature version
P_test = P_test.reshape(360, 24, N_CLASSES)
print(P_test.shape, P_test.min(), P_test.max())              # (360, 24, 29), values between 0 and 1

write_submission("submission_diff.zip", P_test)

(360, 24, 29) 1.8562769266861295e-08 0.9999968057939301


In [44]:
P_sub = smooth(P_test)                  # 1. smoothing over time
P_sub = to_half(P_sub, 0.25)            # 2. cutoff 0.25 moved to 0.5
print(P_sub.shape, P_sub.min(), P_sub.max())     # (360, 24, 29), between 0 and 1

write_submission("submission_smooth_thr.zip", P_sub)

(360, 24, 29) 5.0201882491379096e-08 0.9999471314855097
